# 00 — Data Preparation

**Goal:** produce `data/processed/pancreas_prepped.h5ad`, the single canonical
object every other notebook consumes. Nothing downstream is correct if this isn't.

**Output schema** (asserted at the end of this notebook):

| Field | Contents |
|---|---|
| `X` | log-normalized expression, HVG-subset |
| `layers['counts']` | raw integer counts (scVI + the from-scratch VAE need these) |
| `obs['batch']` | sequencing technology, categorical |
| `obs['cell_type']` | curated annotation, categorical |
| `var['ensembl_id']` | Ensembl gene IDs (Geneformer tokenizer) |
| `obsm['X_pca']` | uncorrected PCA — the control arm + the "before" UMAP |

Cells 3–7 are prototyped here, then lifted into `src/data.py::load_pancreas()` once stable.

In [9]:
# Cell 2 — Setup: make src importable, seed, point scanpy at project dirs.

import sys, os
# Put the repo root on sys.path so `from src...` resolves regardless of kernel CWD.
# Assumes the kernel's working dir is notebooks/ (VS Code's default) → repo root is "..".
sys.path.insert(0, os.path.abspath(".."))

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

from src.utils import set_seeds, DATA_RAW, DATA_PROCESSED, FIGURES
set_seeds(0)

# Point scanpy at the project dirs (mirrors your notebook 01)
sc.settings.datasetdir = str(DATA_RAW)   # scanpy's downloaded datasets land here
sc.settings.figdir     = str(FIGURES)

%load_ext autoreload
%autoreload 2

print("Processed data will be saved to:", DATA_PROCESSED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Processed data will be saved to: E:\Data_Science_Bio\multi-batch-integration\data\processed


In [10]:
# Cell 3 — Load the scIB pancreas benchmark, then interrogate it.
#
# The loader is filled in (verified source). The interrogation is yours: the
# point is to *look* and decide, because every later cell trusts what you conclude.

# ~126 MB download to data/raw on first run; reads locally on every run after.
adata = sc.read(
    DATA_RAW / "pancreas.h5ad",
    backup_url="https://figshare.com/ndownloader/files/24539828",
)
adata   # expect: 16382 x 19093, obs ['tech','celltype','size_factors'], layers ['counts']

# --- Interrogate. Answer three questions before moving on. ---
#
# Q1. BATCH + LABELS: which obs column is the batch, which is the annotation?
#     TODO: value_counts() on 'tech' (how many technologies?) and on 'celltype'
#     (how many cell types?).
#
# Q2. WHAT IS IN X?  The one that matters. counts is in a layer, so X is some
#     processed form — but which?
#     TODO: compare X against layers['counts']. Check X for negative values
#     (negatives => scaled/z-scored), integer vs float, and the min/max of each.
#     Decide: is X log-normalized, scaled, or something else? You're going to
#     recompute it from counts regardless — this just tells you what you're replacing.
#
# Q3. GENE IDS: are var_names gene symbols or Ensembl IDs?
#     TODO: look at adata.var and adata.var_names. Geneformer needs Ensembl IDs,
#     so this tells you how much mapping notebook 00 still owes.

AnnData object with n_obs × n_vars = 16382 × 19093
    obs: 'tech', 'celltype', 'size_factors'
    layers: 'counts'

In [17]:
# Q1. BATCH + LABELS: which obs column is the batch, which is the annotation?
#     TODO: value_counts() on 'tech' (how many technologies?) and on 'celltype'
#     (how many cell types?).

adata.obs.tech.value_counts() # how many tech

tech
inDrop3       3605
smartseq2     2394
celseq2       2285
inDrop1       1937
inDrop2       1724
smarter       1492
inDrop4       1303
celseq        1004
fluidigmc1     638
Name: count, dtype: int64

In [18]:
adata.obs.celltype.value_counts() # how many cell types

celltype
alpha                 5493
beta                  4169
ductal                2142
acinar                1669
delta                 1055
gamma                  699
activated_stellate     464
endothelial            313
quiescent_stellate     193
macrophage              79
mast                    42
epsilon                 32
schwann                 25
t_cell                   7
Name: count, dtype: int64

In [ ]:
pd.crosstab(adata.obs.tech, adata.obs.celltype) # tech vs # celltype

celltype,acinar,activated_stellate,alpha,beta,delta,ductal,endothelial,epsilon,gamma,macrophage,mast,quiescent_stellate,schwann,t_cell
tech,,,,,,,,,,,,,,
celseq,228,19,191,161,50,327,5,1,18,1,1,1,1,0
celseq2,274,90,843,445,203,258,21,4,110,15,6,12,4,0
fluidigmc1,21,16,239,258,25,36,14,1,18,1,3,1,5,0
inDrop1,110,51,236,872,214,120,130,13,70,14,8,92,5,2
inDrop2,3,81,676,371,125,301,23,2,86,17,9,22,6,2
inDrop3,843,100,1130,787,161,376,92,2,36,14,7,54,1,2
inDrop4,2,52,284,495,101,280,7,1,63,10,1,5,1,1
smarter,0,0,886,472,49,0,0,0,85,0,0,0,0,0
smartseq2,188,55,1008,308,127,444,21,8,213,7,7,6,2,0


In [ ]:
# Q2. WHAT IS IN X?  The one that matters. counts is in a layer, so X is some
#     processed form — but which?
#     TODO: compare X against layers['counts']. Check X for negative values
#     (negatives => scaled/z-scored), integer vs float, and the min/max of each.
#     Decide: is X log-normalized, scaled, or something else? You're going to
#     recompute it from counts regardless — this just tells you what you're replacing.

In [ ]:
# Q3. GENE IDS: are var_names gene symbols or Ensembl IDs?
#     TODO: look at adata.var and adata.var_names. Geneformer needs Ensembl IDs,
#     so this tells you how much mapping notebook 00 still owes.